In [0]:
%sql
USE CATALOG `end-to-end-geospatial-data-pipeline-for-urban-encroachment-analytics`;
USE SCHEMA `hydraa`;

In [0]:
%pip install apache-sedona geopandas shapely

from sedona.spark import SedonaContext
from sedona.spark.register.geo_registrator import SedonaRegistrator
from sedona.spark.utils.adapter import Adapter
from pyspark.sql import SparkSession

# Initialize Sedona SQL functions within your active Spark Session
spark = SedonaContext.create(spark)

In [0]:
import os
import json
import requests
import geopandas as gpd
import pandas as pd

# 1. Reference datasets carrying real lake names
#    - Telangana Irrigation Dept waterbodies (OpenCity)
#    - OpenStreetMap named water bodies WITH full geometry (Overpass)
#    - Wikidata lake gazetteer (SPARQL)
BRONZE_DIR = "/Volumes/end-to-end-geospatial-data-pipeline-for-urban-encroachment-analytics/hydraa/bronze"
SILVER_TABLE = "`end-to-end-geospatial-data-pipeline-for-urban-encroachment-analytics`.hydraa.silver_hydra_lakes"
WATERBODIES_URL = "https://data.opencity.in/dataset/c662fa8e-e776-4896-a907-cb4fbad8f89d/resource/6143bd88-8830-49ca-87fd-67ea30abc2da/download/greater_hyd_waterbodies.geojson"
WATERBODIES_PATH = os.path.join(BRONZE_DIR, "greater_hyd_waterbodies.geojson")
OSM_WATERS_PATH = os.path.join(BRONZE_DIR, "osm_named_waters_v2.json")
WIKIDATA_PATH = os.path.join(BRONZE_DIR, "wikidata_lakes.json")

print("📥 Fetching lake name reference datasets...")

# Telangana Irrigation Dept "Waterbodies in Greater Hyderabad" via OpenCity
# (tank names, nearest habitations, districts - with polygon perimeters)
if not os.path.exists(WATERBODIES_PATH):
    response = requests.get(WATERBODIES_URL, timeout=60)
    response.raise_for_status()
    if b"FeatureCollection" not in response.content[:512]:
        raise ValueError("Irrigation waterbodies download is not a GeoJSON FeatureCollection")
    with open(WATERBODIES_PATH, "wb") as f:
        f.write(response.content)
print(f"  ✅ Irrigation waterbodies landed at: {WATERBODIES_PATH}")

# OpenStreetMap named water bodies WITH full geometry (Hussain Sagar, Durgam Cheruvu, ...)
if not os.path.exists(OSM_WATERS_PATH):
    BBOX = "17.15,78.05,17.75,78.75"
    overpass_query = (
        f'[out:json][timeout:120];'
        f'(way["natural"="water"]["name"]({BBOX});'
        f'relation["natural"="water"]["name"]({BBOX});'
        f'way["water"~"pond|reservoir|basin|canal|river|stream"]["name"]({BBOX});'
        f'way["landuse"~"reservoir|basin"]["name"]({BBOX}););'
        f'out geom qt 6000;'
    )
    response = requests.post("https://overpass-api.de/api/interpreter",
                             data={"data": overpass_query}, timeout=180,
                             headers={"User-Agent": "hyderabad-lakes-pipeline/1.0"})
    response.raise_for_status()
    if b"elements" not in response.content[:512]:
        raise ValueError("Overpass response is not the expected JSON")
    with open(OSM_WATERS_PATH, "wb") as f:
        f.write(response.content)
print(f"  ✅ OSM named waters (with geometry) landed at: {OSM_WATERS_PATH}")

# Wikidata lake gazetteer: lakes, reservoirs and tanks within 45 km of Hyderabad
if not os.path.exists(WIKIDATA_PATH):
    sparql = """
    SELECT ?item ?itemLabel ?coord WHERE {
      SERVICE wikibase:around {
        ?item wdt:P625 ?coord .
        bd:serviceParam wikibase:center "Point(78.4867 17.385)"^^geo:wktLiteral .
        bd:serviceParam wikibase:radius "45" .
      }
      VALUES ?class { wd:Q15324 wd:Q131681 wd:Q231307 wd:Q15326 }
      ?item wdt:P31 ?class .
      SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
    } LIMIT 300
    """
    response = requests.get("https://query.wikidata.org/sparql",
                            params={"query": sparql, "format": "json"}, timeout=90,
                            headers={"User-Agent": "hyderabad-lakes-pipeline/1.0"})
    response.raise_for_status()
    if b"bindings" not in response.content[:512]:
        raise ValueError("Wikidata response is not the expected SPARQL JSON")
    with open(WIKIDATA_PATH, "wb") as f:
        f.write(response.content)
print(f"  ✅ Wikidata gazetteer landed at: {WIKIDATA_PATH}")

# 2. Load the irrigation waterbodies into Spark (name, habitation, district, WKT perimeter)
wb = gpd.read_file(WATERBODIES_PATH)
wb = wb[wb.geometry.notna() & ~wb.geometry.is_empty].copy()
for src, dst in [("Tank_Name", "tank_name"), ("HAB_N", "hab_name"), ("DMV_N", "village"), ("D_N", "district")]:
    wb[dst] = wb[src].astype("string").str.strip()
    wb.loc[wb[dst] == "", dst] = None
ref = pd.DataFrame({
    "tank_name": wb["tank_name"],
    "hab_name": wb["hab_name"],
    "village": wb["village"],
    "district": wb["district"],
    "wkt": wb.geometry.apply(lambda g: g.wkt),
})
spark.createDataFrame(ref).createOrReplaceTempView("irrigation_waterbodies")

# 3. Load the OSM named waters: closed ways become polygons for precise overlap
#    matching; relations (multi-part) become centre points for proximity matching
with open(OSM_WATERS_PATH) as f:
    osm_elements = [e for e in json.load(f).get("elements", [])
                    if e.get("tags", {}).get("name")]

osm_polys, osm_centers = [], []
for e in osm_elements:
    name = e["tags"]["name"].strip()
    geom = e.get("geometry") or []
    if e.get("type") == "way" and geom and geom[0] == geom[-1] and len(geom) >= 4:
        ring = ", ".join(f"{p['lon']} {p['lat']}" for p in geom[:-1] + [geom[0]])
        osm_polys.append((name, f"POLYGON(({ring}))"))
    else:
        pts = [p for m in e.get("members", []) for p in (m.get("geometry") or [])] + geom
        if pts:
            osm_centers.append((name,
                                 sum(p["lon"] for p in pts) / len(pts),
                                 sum(p["lat"] for p in pts) / len(pts)))

spark.createDataFrame(osm_polys, "osm_name STRING, wkt STRING").createOrReplaceTempView("osm_named_polygons")
spark.createDataFrame(osm_centers, "osm_name STRING, lon DOUBLE, lat DOUBLE").createOrReplaceTempView("osm_named_centers")
print(f"  🗺️ OSM: {len(osm_polys)} named polygons, {len(osm_centers)} named centres")

# 3b. Load the Wikidata gazetteer (name + coordinate)
with open(WIKIDATA_PATH) as f:
    wd_bindings = json.load(f)["results"]["bindings"]
wd_rows = []
for b in wd_bindings:
    lon, lat = b["coord"]["value"].replace("Point(", "").replace(")", "").split()
    wd_rows.append((b["itemLabel"]["value"].strip(), float(lon), float(lat)))
spark.createDataFrame(wd_rows, "wd_name STRING, lon DOUBLE, lat DOUBLE").createOrReplaceTempView("wikidata_lakes")
print(f"  📚 Wikidata: {len(wd_rows)} gazetteer entries")

# 4. Match every Silver lake polygon to the best available real name.
#    Priority: OSM polygon overlap > OSM centre proximity > Wikidata >
#    Telangana Irrigation Dept tank name (>10% area overlap).
#    Every lake also gets its nearest habitation, village and district so that
#    unnamed tanks still receive a human-readable contextual label.
print("🔎 Matching real names onto lake polygons...")
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW lake_names AS
    WITH lakes AS (
        SELECT id AS lake_id, wkt_geometry,
               ST_Centroid(ST_GeomFromWKT(wkt_geometry)) AS centroid
        FROM {SILVER_TABLE}
    ),
    irrigation_overlap AS (
        SELECT l.lake_id, i.tank_name, i.hab_name, i.village, i.district,
               ROW_NUMBER() OVER (PARTITION BY l.lake_id ORDER BY
                   ST_Area(ST_Intersection(ST_GeomFromWKT(l.wkt_geometry), ST_GeomFromWKT(i.wkt))) DESC) AS rn,
               ST_Area(ST_Intersection(ST_GeomFromWKT(l.wkt_geometry), ST_GeomFromWKT(i.wkt)))
                   / NULLIF(ST_Area(ST_GeomFromWKT(l.wkt_geometry)), 0) AS overlap_ratio
        FROM lakes l JOIN irrigation_waterbodies i
          ON ST_Intersects(ST_GeomFromWKT(l.wkt_geometry), ST_GeomFromWKT(i.wkt))
    ),
    irrigation_nearest AS (
        SELECT l.lake_id, i.hab_name, i.village, i.district,
               ROW_NUMBER() OVER (PARTITION BY l.lake_id ORDER BY
                   ST_Distance(l.centroid, ST_Centroid(ST_GeomFromWKT(i.wkt)))) AS rn
        FROM lakes l JOIN irrigation_waterbodies i
          ON ST_Distance(l.centroid, ST_Centroid(ST_GeomFromWKT(i.wkt))) < 0.03
    ),
    osm_poly_match AS (
        SELECT l.lake_id, o.osm_name,
               ROW_NUMBER() OVER (PARTITION BY l.lake_id ORDER BY
                   ST_Area(ST_Intersection(ST_GeomFromWKT(l.wkt_geometry), ST_GeomFromWKT(o.wkt))) DESC) AS rn,
               ST_Area(ST_Intersection(ST_GeomFromWKT(l.wkt_geometry), ST_GeomFromWKT(o.wkt)))
                   / NULLIF(ST_Area(ST_GeomFromWKT(l.wkt_geometry)), 0) AS overlap_ratio,
               ST_Contains(ST_GeomFromWKT(l.wkt_geometry), ST_Centroid(ST_GeomFromWKT(o.wkt))) AS osm_inside_lake
        FROM lakes l JOIN osm_named_polygons o
          ON ST_Intersects(ST_GeomFromWKT(l.wkt_geometry), ST_GeomFromWKT(o.wkt))
    ),
    osm_center_match AS (
        SELECT l.lake_id, c.osm_name,
               ROW_NUMBER() OVER (PARTITION BY l.lake_id ORDER BY
                   ST_Distance(ST_GeomFromWKT(l.wkt_geometry), ST_Point(c.lon, c.lat))) AS rn
        FROM lakes l JOIN osm_named_centers c
          ON ST_Distance(ST_GeomFromWKT(l.wkt_geometry), ST_Point(c.lon, c.lat)) < 0.0045
    ),
    wikidata_match AS (
        SELECT l.lake_id, w.wd_name,
               ROW_NUMBER() OVER (PARTITION BY l.lake_id ORDER BY
                   ST_Distance(ST_GeomFromWKT(l.wkt_geometry), ST_Point(w.lon, w.lat))) AS rn
        FROM lakes l JOIN wikidata_lakes w
          ON ST_Distance(ST_GeomFromWKT(l.wkt_geometry), ST_Point(w.lon, w.lat)) < 0.0045
    )
    SELECT l.lake_id,
           COALESCE(
               CASE WHEN p.overlap_ratio > 0.02 OR p.osm_inside_lake THEN p.osm_name END,
               c.osm_name,
               w.wd_name,
               CASE WHEN io.overlap_ratio > 0.10 THEN io.tank_name END
           ) AS lake_name,
           CASE WHEN (p.overlap_ratio > 0.02 OR p.osm_inside_lake) AND p.osm_name IS NOT NULL THEN 'OpenStreetMap'
                WHEN c.osm_name IS NOT NULL THEN 'OpenStreetMap'
                WHEN w.wd_name IS NOT NULL THEN 'Wikidata'
                WHEN io.overlap_ratio > 0.10 AND io.tank_name IS NOT NULL THEN 'Telangana Irrigation Dept'
           END AS name_source,
           COALESCE(io.district, inr.district) AS district,
           COALESCE(io.hab_name, inr.hab_name) AS nearest_habitation,
           COALESCE(io.village, inr.village) AS nearest_village
    FROM lakes l
    LEFT JOIN (SELECT * FROM irrigation_overlap WHERE rn = 1) io ON io.lake_id = l.lake_id
    LEFT JOIN (SELECT * FROM irrigation_nearest WHERE rn = 1) inr ON inr.lake_id = l.lake_id
    LEFT JOIN (SELECT * FROM osm_poly_match WHERE rn = 1) p ON p.lake_id = l.lake_id
    LEFT JOIN (SELECT * FROM osm_center_match WHERE rn = 1) c ON c.lake_id = l.lake_id
    LEFT JOIN (SELECT * FROM wikidata_match WHERE rn = 1) w ON w.lake_id = l.lake_id
""")

stats = spark.sql("""
    SELECT COUNT(*) AS lakes,
           COUNT(lake_name) AS with_real_name,
           COUNT(CASE WHEN district IS NOT NULL OR nearest_habitation IS NOT NULL OR nearest_village IS NOT NULL THEN 1 END) AS with_context
    FROM lake_names
""").collect()[0]
print(f"🏷️ Real names: {stats['with_real_name']} | with habitation/district context: {stats['with_context']} | lakes: {stats['lakes']}")

In [0]:
from pyspark.sql.functions import col

# 1. Sedona spatial SQL functions are already registered on this Spark session
#    by SedonaContext.create(spark) in the previous cell (the legacy
#    SedonaRegistrator path requires direct JVM access, which serverless compute does not allow)

print("⚡ Fetching Silver records and initializing spatial engine...")

# 2. Define the buffer distance in decimal degrees (~30 meters)
BUFFER_DEGREES = 0.00027
GOLD_TABLE_NAME = "gold_hydra_buffers"

# 3. Join the real lake names (matched in the enrichment cell) onto each Silver record,
#    then use Spark SQL and Sedona to generate the buffer polygons.
#    ST_GeomFromWKT converts our text string back into a real spatial object
#    ST_Buffer blows the polygon outwards by our specified distance
#    ST_AsText converts it back to a clean string format for storage
gold_df = spark.sql(f"""
    WITH enriched AS (
        SELECT
            s.id AS lake_id,
            s.pi_code,
            s.descr_2 AS waterbody_type,
            s.wkt_geometry,
            n.lake_name,
            n.name_source,
            n.district,
            n.nearest_habitation,
            n.nearest_village,
            -- GHMC survey note used as a fallback name only when it is a place name,
            -- not an encroachment annotation like "Mining Inside Tank"
            CASE
                WHEN s.pi_code IS NULL OR s.pi_code = '' THEN NULL
                WHEN s.pi_code RLIKE '(?i)(inside|in tank|crop|mining|quarry|built)' THEN NULL
                ELSE s.pi_code
            END AS survey_name,
            -- Standardised encroachment flag from the GHMC survey note
            CASE
                WHEN s.pi_code RLIKE '(?i)mining' THEN 'Mining inside FTL'
                WHEN s.pi_code RLIKE '(?i)quarry' THEN 'Quarry inside FTL'
                WHEN s.pi_code RLIKE '(?i)built' THEN 'Built-up inside FTL'
                WHEN s.pi_code RLIKE '(?i)crop' THEN 'Crop cultivation inside FTL'
            END AS ghmc_encroachment_flag,
            -- Human-readable fallback label so no lake is ever an opaque feature id
            CASE
                WHEN (NULLIF(n.nearest_habitation, '') IS NOT NULL OR NULLIF(n.nearest_village, '') IS NOT NULL) AND n.district IS NOT NULL THEN
                    CONCAT('Unnamed tank near ', COALESCE(NULLIF(n.nearest_habitation, ''), NULLIF(n.nearest_village, '')), ', ', n.district)
                WHEN NULLIF(n.nearest_habitation, '') IS NOT NULL OR NULLIF(n.nearest_village, '') IS NOT NULL THEN
                    CONCAT('Unnamed tank near ', COALESCE(NULLIF(n.nearest_habitation, ''), NULLIF(n.nearest_village, '')))
                WHEN n.district IS NOT NULL THEN CONCAT('Unnamed tank in ', n.district)
                ELSE 'Unnamed tank in Greater Hyderabad'
            END AS contextual_label
        FROM `end-to-end-geospatial-data-pipeline-for-urban-encroachment-analytics`.hydraa.silver_hydra_lakes s
        LEFT JOIN lake_names n ON n.lake_id = s.id
    )
    SELECT
        lake_id,
        COALESCE(lake_name, survey_name, contextual_label, lake_id) AS lake_name,
        COALESCE(
            name_source,
            CASE WHEN survey_name IS NOT NULL THEN 'GHMC survey attribute'
                 WHEN contextual_label IS NOT NULL THEN 'Nearest habitation (contextual)'
                 ELSE 'Feature ID' END
        ) AS name_source,
        pi_code AS local_name,
        waterbody_type,
        district,
        nearest_habitation,
        nearest_village,
        ghmc_encroachment_flag,
        -- Approximate areas: decimal degrees squared converted at Hyderabad's latitude (~17.4N)
        ROUND(ST_Area(ST_GeomFromWKT(wkt_geometry)) * 11756.0, 4) AS lake_area_km2,
        ST_X(ST_Centroid(ST_GeomFromWKT(wkt_geometry))) AS centroid_lon,
        ST_Y(ST_Centroid(ST_GeomFromWKT(wkt_geometry))) AS centroid_lat,
        wkt_geometry AS ftl_geometry_wkt,
        ST_AsText(ST_Buffer(ST_GeomFromWKT(wkt_geometry), {BUFFER_DEGREES})) AS buffer_geometry_wkt
    FROM enriched
""")

print("📐 Calculation complete. Writing to Gold Delta Table...")

# 4. Save the final high-value dataset
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_TABLE_NAME)

print(f"✅ Gold Layer successfully deployed to table: {GOLD_TABLE_NAME}!")


In [0]:
import os
import gzip
import json
import time
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
from shapely import wkt as shapely_wkt

# 1. Configuration: buildings are fetched around every lake in batches and
#    cached (gzipped) in the bronze volume, so re-runs do not hit Overpass again
BRONZE_DIR = "/Volumes/end-to-end-geospatial-data-pipeline-for-urban-encroachment-analytics/hydraa/bronze"
BATCH_CACHE_DIR = os.path.join(BRONZE_DIR, "osm_building_batches")
os.makedirs(BATCH_CACHE_DIR, exist_ok=True)
BUFFER_DEGREES = 0.00027
GOLD_ENCROACHMENT_TABLE = "gold_hydra_encroachment"
GOLD_BUILDINGS_TABLE = "gold_hydra_buffer_buildings"

print("🏗️ Encroachment analysis: OSM buildings inside protected lake zones...")

# 2. Load the Silver lake geometries and compute a fetch anchor per lake
#    (anchor point + covering radius so the buffer zone is fully included)
silver = spark.sql(
    "SELECT id, wkt_geometry FROM `end-to-end-geospatial-data-pipeline-for-urban-encroachment-analytics`.hydraa.silver_hydra_lakes"
).toPandas()
silver["geometry"] = silver["wkt_geometry"].apply(shapely_wkt.loads)

DEG_M = 111320.0
anchors = []
for _, row in silver.iterrows():
    g = row["geometry"]
    cx, cy = g.centroid.x, g.centroid.y
    if g.geom_type == "Polygon":
        pts = np.array(g.exterior.coords)
    else:
        pts = np.vstack([p.exterior.coords for p in g.geoms])
    d = np.sqrt(((pts[:, 0] - cx) * DEG_M * np.cos(np.radians(cy))) ** 2 + ((pts[:, 1] - cy) * DEG_M) ** 2)
    anchors.append((row["id"], cy, cx, g, float(d.max()) + 80))

# 3. Fetch OSM building footprints around every lake (cached per batch).
#    Queries are kept small (<=20 lakes), rotated across two Overpass endpoints,
#    retried with backoff, and halved on persistent failure.
OVERPASS_ENDPOINTS = ["https://overpass-api.de/api/interpreter",
                      "https://overpass.kumi.systems/api/interpreter"]

def fetch_batch_geo(batch):
    if not batch:
        return []
    parts = "".join(f'way["building"](around:{int(a[4])},{a[1]:.6f},{a[2]:.6f});' for a in batch)
    query = f'[out:json][timeout:180];({parts});out geom qt;'
    last_error = None
    for attempt in range(6):
        try:
            response = requests.post(OVERPASS_ENDPOINTS[attempt % 2],
                                     data={"data": query}, timeout=240,
                                     headers={"User-Agent": "hyderabad-lakes-pipeline/1.0"})
            if response.status_code in (429, 502, 503, 504):     # throttle/overload - back off
                last_error = f"HTTP {response.status_code}"
                time.sleep(20 * (attempt + 1))
                continue
            response.raise_for_status()
            if b"elements" not in response.content[:512]:
                raise ValueError("Unexpected Overpass response")
            return response.json().get("elements", [])
        except Exception as e:
            last_error = e
            time.sleep(15 * (attempt + 1))
    if len(batch) == 1:                                          # give up on a single lake
        print(f"  ⚠️ lake {batch[0][0]} skipped after retries ({last_error})")
        return []
    mid = len(batch) // 2                                        # split and try smaller
    return fetch_batch_geo(batch[:mid]) + fetch_batch_geo(batch[mid:])

BATCH_SIZE = 20
batches = [anchors[i:i + BATCH_SIZE] for i in range(0, len(anchors), BATCH_SIZE)]

def fetch_buildings(batch, idx):
    cache = os.path.join(BATCH_CACHE_DIR, f"batch_{idx:03d}.json.gz")
    if os.path.exists(cache):
        with gzip.open(cache, "rt") as f:
            return json.load(f).get("elements", [])
    elements = fetch_batch_geo(batch)
    with gzip.open(cache, "wt") as f:
        json.dump({"elements": elements}, f)
    return elements

candidate_rows = []
for idx, batch in enumerate(batches):
    elements = fetch_buildings(batch, idx)
    buildings = []
    for e in elements:
        g = e.get("geometry")
        if g and g[0] == g[-1] and len(g) >= 4:             # closed building footprints only
            buildings.append((e["id"], e.get("tags", {}).get("building", "yes"),
                              Polygon([(p["lon"], p["lat"]) for p in g[:-1]])))
    if buildings:
        bld = gpd.GeoDataFrame(buildings, columns=["osm_id", "building_type", "geometry"])
        for lake_id, _, _, lake_geom, _ in batch:
            ring_geom = lake_geom.buffer(BUFFER_DEGREES).difference(lake_geom)   # the protected strip
            in_ring = bld.geometry.intersects(ring_geom)
            on_water = bld.geometry.intersects(lake_geom)
            for osm_id, btype in zip(bld.loc[in_ring, "osm_id"], bld.loc[in_ring, "building_type"]):
                candidate_rows.append((lake_id, int(osm_id), btype, "buffer_ring"))
            for osm_id, btype in zip(bld.loc[on_water, "osm_id"], bld.loc[on_water, "building_type"]):
                candidate_rows.append((lake_id, int(osm_id), btype, "on_water"))
    print(f"  batch {idx + 1}/{len(batches)}: {len(buildings)} buildings fetched")
    time.sleep(2)                                          # be polite to the Overpass API

# 4. De-duplicate (overlapping lake radii can return the same building twice)
if candidate_rows:
    candidates = pd.DataFrame(candidate_rows, columns=["lake_id", "osm_id", "building_type", "zone"]).drop_duplicates()
    spark.createDataFrame(candidates).createOrReplaceTempView("buffer_buildings")
else:
    spark.sql("CREATE OR REPLACE TEMP VIEW buffer_buildings AS "
              "SELECT CAST(NULL AS STRING) lake_id, CAST(NULL AS LONG) osm_id, "
              "CAST(NULL AS STRING) building_type, CAST(NULL AS STRING) zone WHERE 1=0")
print(f"🏠 Structures inside protected zones: {candidates.shape[0] if candidate_rows else 0} "
      f"across {candidates.lake_id.nunique() if candidate_rows else 0} lakes")

# 5. Per-lake encroachment summary: OSM building pressure + GHMC survey flag -> status & score
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW lake_encroachment AS
    WITH per_lake AS (
        SELECT lake_id,
               SUM(CASE WHEN zone = 'buffer_ring' THEN 1 ELSE 0 END) AS buildings_in_buffer_ring,
               SUM(CASE WHEN zone = 'on_water' THEN 1 ELSE 0 END) AS buildings_on_water
        FROM buffer_buildings
        GROUP BY lake_id
    )
    SELECT g.lake_id, g.lake_name, g.district, g.lake_area_km2, g.ghmc_encroachment_flag,
           COALESCE(p.buildings_in_buffer_ring, 0) AS buildings_in_buffer_ring,
           COALESCE(p.buildings_on_water, 0) AS buildings_on_water,
           COALESCE(p.buildings_in_buffer_ring, 0)
               + 3 * COALESCE(p.buildings_on_water, 0)
               + CASE WHEN g.ghmc_encroachment_flag IS NOT NULL THEN 25 ELSE 0 END AS encroachment_score,
           CASE
               WHEN g.ghmc_encroachment_flag IS NOT NULL
                    AND (COALESCE(p.buildings_in_buffer_ring, 0) + COALESCE(p.buildings_on_water, 0)) > 0 THEN 'Confirmed (survey + structures)'
               WHEN g.ghmc_encroachment_flag IS NOT NULL THEN 'Reported (GHMC survey)'
               WHEN COALESCE(p.buildings_on_water, 0) > 0 THEN 'Structures on the water'
               WHEN COALESCE(p.buildings_in_buffer_ring, 0) >= 20 THEN 'High building pressure'
               WHEN COALESCE(p.buildings_in_buffer_ring, 0) > 0 THEN 'Some building pressure'
               ELSE 'No pressure detected'
           END AS encroachment_status
    FROM `end-to-end-geospatial-data-pipeline-for-urban-encroachment-analytics`.hydraa.gold_hydra_buffers g
    LEFT JOIN per_lake p ON p.lake_id = g.lake_id
""")

spark.table("lake_encroachment").write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_ENCROACHMENT_TABLE)

spark.table("buffer_buildings").write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(GOLD_BUILDINGS_TABLE)

print(f"✅ Encroachment tables deployed: {GOLD_ENCROACHMENT_TABLE}, {GOLD_BUILDINGS_TABLE}")

kpis = spark.sql(f"""
    SELECT COUNT(*) AS lakes,
           SUM(CASE WHEN ghmc_encroachment_flag IS NOT NULL THEN 1 ELSE 0 END) AS reported_by_ghmc,
           SUM(CASE WHEN buildings_on_water > 0 THEN 1 ELSE 0 END) AS with_structures_on_water,
           SUM(CASE WHEN buildings_in_buffer_ring > 0 THEN 1 ELSE 0 END) AS with_buffer_pressure
    FROM `end-to-end-geospatial-data-pipeline-for-urban-encroachment-analytics`.hydraa.{GOLD_ENCROACHMENT_TABLE}
""").collect()[0]
print(f"📊 Lakes: {kpis['lakes']} | GHMC-reported encroachment: {kpis['reported_by_ghmc']} | "
      f"structures on the water: {kpis['with_structures_on_water']} | building pressure in buffer zone: {kpis['with_buffer_pressure']}")

In [0]:
%sql
-- Let's check a few lakes and see the difference in boundary descriptions
SELECT 
    local_name,
    lake_name,
    waterbody_type,
    substring(ftl_geometry_wkt, 1, 50) as ftl_sample,
    substring(buffer_geometry_wkt, 1, 50) as buffer_sample
FROM 
    hydraa.gold_hydra_buffers 
-- WHERE 
--     local_name IS NOT NULL 
-- LIMIT 5;
